## Cell 1: Environment Setup & Imports

In [1]:
import os

import yt_dlp
from dotenv import load_dotenv

# Load environment variables from the .env file located in the parent directory
load_dotenv(dotenv_path="../.env")

print("✅ Libraries imported and environment variables loaded.")

✅ Libraries imported and environment variables loaded.


## Cell 2: Ingestion Function Definition

## `extract_youtube_id(url)`
Parses a YouTube URL and extracts the unique video ID.  
Supports both standard (`?v=`) and shortened (`youtu.be/...`) formats.

### `download_youtube_audio(url)`
Downloads the audio track of a YouTube video as an MP3.  
Uses the video ID as the filename and stores the file in `../data/`.  
Returns the video ID and the path to the saved MP3.

In [2]:
## Cell 2: Ingestion Function Definition

import urllib.parse as urlparse


def extract_youtube_id(url: str) -> str:
    """
    Extract the unique YouTube video ID from either a standard
    or shortened YouTube URL.

    Args:
        url: YouTube video URL.

    Returns:
        The YouTube video ID.

    Raises:
        ValueError: If the URL is not a valid YouTube video URL.
    """
    parsed_url = urlparse.urlparse(url)
    query_params = urlparse.parse_qs(parsed_url.query)

    if "v" in query_params:
        return query_params["v"][0]

    if parsed_url.hostname == "youtu.be":
        return parsed_url.path.lstrip("/")

    raise ValueError("Invalid YouTube URL provided.")


def download_youtube_audio(url: str) -> tuple[str, str]:
    """
    Download the audio track from a YouTube video and save it as an MP3.

    The downloaded file is stored inside the ../data directory using
    the YouTube video ID as its filename.

    Args:
        url: YouTube video URL.

    Returns:
        A tuple containing:
            - video_id: Unique YouTube video identifier.
            - audio_path: Path to the downloaded MP3 file.
    """

    # Step 1: Extract the unique video ID
    video_id = extract_youtube_id(url)

    print(f"🎯 Video ID: {video_id}")
    print("⏳ Downloading audio...")

    # Step 2: Ensure the output directory exists
    output_dir = "../data"
    os.makedirs(output_dir, exist_ok=True)

    # Step 3: Configure yt-dlp
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": os.path.join(output_dir, f"{video_id}.%(ext)s"),
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }
        ],
        "quiet": False,
    }

    # Step 4: Download the audio
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:  # type: ignore
        ydl.download([url])

    audio_path = os.path.join(output_dir, f"{video_id}.mp3")

    print("\n✅ Audio downloaded successfully.")
    print(f"📁 Saved to: {os.path.abspath(audio_path)}")

    return video_id, audio_path

## Ingestion Pipeline Execution

### User Input & Validation
Prompts the user for a YouTube URL and ensures a non‑empty value before running the pipeline.

### Audio Download
Calls the ingestion function to:
- Extract the video ID  
- Download the YouTube audio as an MP3  
- Store the file path and ID

### Session State Tracking
Creates a `session_state.json` file containing the active video ID so the pipeline can remember which video is currently being processed.

### Output
Prints confirmation messages showing:
- The active video ID  
- The saved audio file path  
- The location of the session state file

If no URL is provided, it stops and alerts the user.


In [3]:
## Cell 3: Execute the Ingestion Pipeline

import json

# Prompt the user for a YouTube URL
youtube_url = input("🔗 Please enter the YouTube URL: ").strip()

if youtube_url:

    # Download the audio and retrieve the video ID
    video_id, audio_path = download_youtube_audio(youtube_url)

    # Save the active pipeline state
    session_state = {
        "active_video_id": video_id
    }

    state_path = "../data/session_state.json"

    with open(state_path, "w", encoding="utf-8") as file:
        json.dump(session_state, file, indent=4)

    print("\n✅ Ingestion completed successfully.")
    print(f"🎥 Active Video ID: {video_id}")
    print(f"📁 Audio file: {audio_path}")
    print(f"💾 Pipeline state saved to: {state_path}")

else:
    print("❌ No YouTube URL was provided. Please run the cell again.")

🎯 Video ID: 4_oufjP2yeM
⏳ Downloading audio...
[youtube] Extracting URL: https://www.youtube.com/watch?v=4_oufjP2yeM
[youtube] 4_oufjP2yeM: Downloading webpage


[youtube] 4_oufjP2yeM: Downloading android vr player API JSON
[info] 4_oufjP2yeM: Downloading 1 format(s): 251
[download] Destination: ..\data\4_oufjP2yeM.webm
[download] 100% of   10.41MiB in 00:00:00 at 28.37MiB/s    
[ExtractAudio] Destination: ..\data\4_oufjP2yeM.mp3
Deleting original file ..\data\4_oufjP2yeM.webm (pass -k to keep)

✅ Audio downloaded successfully.
📁 Saved to: c:\Users\con2m\Desktop\IRONHACKCOURSE\FINAL_PROYECT\resumazing-final\data\4_oufjP2yeM.mp3

✅ Ingestion completed successfully.
🎥 Active Video ID: 4_oufjP2yeM
📁 Audio file: ../data\4_oufjP2yeM.mp3
💾 Pipeline state saved to: ../data/session_state.json


# Notebook 1 — Ingestion Summary

This notebook implements a simple ingestion pipeline for YouTube audio.

## 1. URL Parsing
Defines a function to extract the YouTube video ID from both standard and shortened URLs.  
Ensures the URL is valid before continuing.

## 2. Audio Download
Uses `yt-dlp` to download the audio track of the YouTube video as an MP3.  
Stores the file in `../data/` using the video ID as the filename.

## 3. Pipeline Execution
Prompts the user for a YouTube URL, runs the download process, and saves a `session_state.json` file containing the active video ID.  
Provides console feedback showing the downloaded file and saved state.

## Overall
The notebook prepares raw audio data from YouTube and records the current processing context, forming the first step of the full pipeline.
